In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime
from torchsummary import summary
import json
import time
import glob

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Training Dataset Import

In [ ]:
relative_folder = '../../../Data/Datasets/2. Sensor 2/Training Dataset/'
abs_folder = os.path.abspath(relative_folder)
files = os.listdir(abs_folder)
target_file = files[0] 
full_path = os.path.join(abs_folder, target_file)
data = pd.read_csv(full_path, header=0)
data = data.sample(frac=1)

In [ ]:
# Prepare input and output tensors
x_train = torch.tensor(data.iloc[:, :16].values, dtype=torch.float32)
y_train = torch.tensor(data.iloc[:, 16:20].values, dtype=torch.float32)

In [ ]:
input=data.iloc[:,0:16].values
output=data.iloc[:,16:20].values

In [ ]:
x_train = torch.Tensor(input)
y_train = torch.Tensor(output)

In [ ]:
y_train.size()

In [ ]:
nan_indices_x = torch.isnan(x_train).any(dim=1)
x_train = x_train[~nan_indices_x]
y_train = y_train[~nan_indices_x]
nan_indices_y = torch.isnan(y_train).any(dim=1)
x_train = x_train[~nan_indices_y]
y_train = y_train[~nan_indices_y]
mean_input_train = x_train.mean().item()
mean_input_train = torch.tensor(mean_input_train)
std_input_train = x_train.std().item()
std_input_train = torch.tensor(std_input_train)
x_train = x_train.view((x_train.shape[0], 4, 4, ))
input_transform = transforms.Normalize(mean=[mean_input_train], std=[std_input_train])
x_train_normalized = input_transform(x_train)
mean_y = y_train[:, :3].mean(dim=0)
mean_y = torch.tensor(mean_y)
std_y = y_train[:, :3].std(dim=0)
std_y = torch.tensor(std_y)
y_train_normalized = (y_train[:, :3] - mean_y) / std_y
y_train_normalized = torch.cat((y_train_normalized, y_train[:, 3].unsqueeze(1)), dim=1)

In [ ]:
# Reshape to a 4x4 capacitive image
x_train = x_train.view((x_train.shape[0], 4, 4, ))

In [ ]:
# verify sizes
x_train.size()

In [ ]:
cwd = os.getcwd()
print(f"Current Working Directory: {cwd}")
project_root = os.path.abspath(os.path.join(cwd, "../../../"))
print(f"Project Root (calculated): {project_root}")

if os.path.exists(project_root):
    print("Contents of Root:", os.listdir(project_root))
else:
    print("Error: The path '../../../' does not lead to a valid folder from here.")

In [ ]:
y_train_normalized.size()

In [ ]:
x_train_normalized[119]

In [ ]:
x_train_normalized.size()

In [ ]:
x_train_normalized = x_train_normalized.to(device)
y_train_normalized = y_train_normalized.to(device)

In [ ]:
# Create PyTorch datasets and data loaders
train_dataset = TensorDataset(x_train_normalized, y_train_normalized)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, drop_last=True)

## Load Model Architecture

In [ ]:
class CNNRegression(nn.Module):
    def __init__(self):
        super(CNNRegression, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(4)
        
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(8)
        
        self.conv3 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(16)
        
        self.conv4 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(32)

        self.conv5 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        self.fc1 = nn.Linear(in_features=64*4*4, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=4)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        
        x = self.conv1(x)
        x = self.bn1(x)
        x = nn.functional.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = nn.functional.relu(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = nn.functional.relu(x)
        
        x = self.conv4(x)
        x = self.bn4(x)
        x = nn.functional.relu(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = nn.functional.relu(x)
        
        x = x.view(x.size(0), -1)
        
        x = self.fc1(x)
        x = nn.functional.relu(x)
        
        x = self.fc2(x)
        
        return x

In [ ]:
# Instantiate the model
model = CNNRegression()

# Generate a random 4x4 input tensor
input_tensor = torch.rand(4, 4)

# Run one iteration
output_tensor = model(input_tensor.unsqueeze(0))  # Unsqueezing to add batch dimension

# Print the output tensor
print("Output tensor size:", output_tensor.size())
print("Output tensor:", output_tensor)

In [ ]:
model = CNNRegression()
model = model.to(device)

In [ ]:
# Print the summary of the model
summary(model, (4, 4))

In [ ]:
data_parallel_model = torch.nn.DataParallel(model)

## Load Validation Dataset

In [ ]:
relative_folder = '../../../Data/Datasets/2. Sensor 2/Validation Dataset/'
abs_folder = os.path.abspath(relative_folder)
files = os.listdir(abs_folder)
target_file = files[0] 
full_path = os.path.join(abs_folder, target_file)
data_val = pd.read_csv(full_path, header=0)
data_val = data_val.sample(frac=1)
input_val =data_val.iloc[:,0:16].values
output_val=data_val.iloc[:,16:20].values
x_val = torch.Tensor(input_val)
y_val = torch.Tensor(output_val)

In [ ]:
x_val.size()

In [ ]:
# Check for NaN in x_train
nan_indices_x = torch.isnan(x_val).any(dim=1)

# Remove rows with NaN from both x_train_normalized and y_val
x_val = x_val[~nan_indices_x]
y_val = y_val[~nan_indices_x]

# Check for NaN in y_train 
nan_indices_y = torch.isnan(y_val).any(dim=1)
x_val = x_val[~nan_indices_y]
y_val = y_val[~nan_indices_y]

In [ ]:
x_val = x_val.view((x_val.shape[0], 4, 4, ))
x_val = x_val.to(device)
y_val = y_val.to(device)
val_dataset = TensorDataset(x_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, drop_last=True)

## Run the training

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(data_parallel_model.parameters(), lr=0.001, weight_decay=1e-5)
reg_mse_loss = nn.MSELoss()
class_bce_loss = nn.BCEWithLogitsLoss()
reg_weight_location = 0.3
reg_weight_force = 0.5
reg_weight_classification = 1 - reg_weight_location - reg_weight_force
val_loss_x = []
val_loss_y = []
val_loss_force = []
# Train the model
for epoch in range(10):
    for i, (inputs, targets) in enumerate(train_loader):
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs =  data_parallel_model(inputs)
        loss_regression_location = reg_mse_loss(outputs[:,0:2], targets[:,0:2])
        loss_regression_force = reg_mse_loss(outputs[:,2:3], targets[:,2:3])
        loss_classification = class_bce_loss(outputs[:,3:4],targets[:,3:4])

        # Backward pass and optimization
        combined_loss = reg_weight_location * loss_regression_location + reg_weight_force * loss_regression_force + reg_weight_classification * loss_classification
        combined_loss.backward()
        optimizer.step()

        # Print training statistics
        if (i + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{10}], Step [{i + 1}/{len(train_loader)}], Loss: {combined_loss.item():.4f}')

    # Validation loop
    with torch.no_grad():
        # Loop for calculating validation loss
        for name, idx, val_loss_list in [("X Location in mm", 0, val_loss_x), 
                                          ("Y Location in mm", 1, val_loss_y), 
                                          ("Force Magnitude in N", 2, val_loss_force)]:
            total_loss = 0
            total_instances = 0  # Total instances where the condition is met
            total_instances_with_condition = 0  # Total instances where the condition is met or not met
            
            # Loop through validation data
            for inputs, targets in val_loader:
                # Normalize inputs
                inputs_normalized = (inputs - mean_input_train) / std_input_train
                
                # Pass through model
                outputs = data_parallel_model(inputs_normalized)
                mean_y_gpu = mean_y.to(outputs.device)
                std_y_gpu = std_y.to(outputs.device)
                
                # Unnormalize outputs
                outputs_unnormalized = outputs[:, :3] * std_y_gpu + mean_y_gpu

                # Concatenate the contact/no-contact column of outputs with the unnormalized outputs
                outputs_unnormalized = torch.cat((outputs_unnormalized, outputs[:, 3].unsqueeze(1)), dim=1)
                
                # Calculate MAE for instances where the force condition is met
                condition_mask = targets[:, 2] < -0.02
                relevant_outputs = outputs_unnormalized[condition_mask]
                relevant_targets = targets[condition_mask]
                
                if relevant_outputs.size(0) > 0:
                    loss = torch.mean(torch.abs(relevant_outputs[:, idx] - relevant_targets[:, idx]))
                    total_loss += loss.item() * relevant_outputs.size(0)  # Accumulate the loss
                    total_instances_with_condition += relevant_outputs.size(0)
                    
                total_instances += targets.size(0)

            if total_instances_with_condition > 0:
                mean_loss = total_loss / total_instances_with_condition
                print(f'Val Loss for {name} (MAE): {mean_loss:.4f}')
                val_loss_list.append(mean_loss)  # Append the validation loss for the current epoch
            else:
                print(f'No instances found where the condition is met for {name}')
    
# Plotting the validation losses
epochs = range(1, 11) 
plt.plot(epochs, val_loss_x, label='X Location in mm')
plt.plot(epochs, val_loss_y, label='Y Location in mm')
plt.plot(epochs, val_loss_force, label='Force Magnitude in N')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss (MAE)')
plt.title('Validation Loss for Different Outputs')
plt.legend()
plt.show()

In [ ]:
# Save model at desired path location
torch.save(data_parallel_model.module.state_dict(),'../../../Data/Trained Models/Single Contact Estimation - Trained Model/Single_Point_Estimation_CNN_'+ datetime.now().strftime("%Y%m%d-%H_%M") + '.pt' )
time.sleep(5)

In [ ]:
# Saving model results on Test dataset

# 1. Get the current directory and find the Project Root
current_folder = os.getcwd()
project_root = os.path.abspath(os.path.join(current_folder, "..", "..", ".."))

# 2. Define the path to the Datasets folder
dataset_dir = os.path.join(project_root, "Data", "Datasets", "2. Sensor 2", "Test Dataset")

# 3. Use glob to find the .txt file automatically
txt_files = glob.glob(os.path.join(dataset_dir, "*.txt"))

# Pick the .txt file found
data_test_path = txt_files[0]
print(f"Loading dataset from: {data_test_path}")
# 4. Load the dataset
# It's comma-separated
data_test = pd.read_csv(data_test_path)
print(data_test.info)
data_test = data_test.sample(frac=1)
input_test =data_test.iloc[:,0:16].values
output_test = data_test.iloc[:,16:20].values
x_test = torch.Tensor(input_test)
y_test = torch.Tensor(output_test)
x_test.size()
# Check for NaN in x_test
nan_indices_x = torch.isnan(x_test).any(dim=1)
# Remove rows with NaN from both x_test and y_test
x_test = x_test[~nan_indices_x]
y_test = y_test[~nan_indices_x]
# Check for NaN in y_test
nan_indices_y = torch.isnan(y_test).any(dim=1)
x_test = x_test[~nan_indices_y]
y_test = y_test[~nan_indices_y]
x_test = x_test.view((x_test.shape[0], 4, 4, ))
x_test = x_test.to(device)
y_test = y_test.to(device)
test_dataset = TensorDataset(x_test, y_test)

test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=True)
dataframe = pd.DataFrame()
reg_mse_loss = nn.MSELoss(reduce=False)
reg_mae_loss = nn.L1Loss(reduce=False)
class_bce_loss = nn.BCEWithLogitsLoss(reduce=False)
m = nn.Sigmoid()
# Define column names in a list
column_names = ["X_Target [mm]", "Y_Target [mm]", "Force_Target [N]", "Touch Classification Target", "X_Pred [mm]", "Y_Pred [mm]", "Force_Pred [N]", "Touch Classification Pred", "Error_X [mm]", "Error_Y [mm]", "Error_Force [N]", "Error_Classification"]
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs_normalized = (inputs - mean_input_train) / std_input_train
        outputs = model(inputs_normalized.float())
        mean_y_gpu = mean_y.to(outputs.device)
        std_y_gpu = std_y.to(outputs.device)    
        outputs_unnormalized = outputs.clone()
        outputs_unnormalized[:, :3] = outputs_unnormalized[:, :3] * std_y_gpu[:3] + mean_y_gpu[:3]
        x_loss = reg_mae_loss(outputs_unnormalized[:,0], targets[:,0])
        y_loss = reg_mae_loss(outputs_unnormalized[:,1], targets[:,1])
        force_loss = reg_mae_loss(outputs_unnormalized[:,2], targets[:,2])
        touch_loss = class_bce_loss(outputs[:,3], targets[:,3])
        outputs[:,3] = m(outputs[:,3])
        outputs_unnormalized[:,3] = outputs[:,3]
        data_row = torch.cat((targets, outputs_unnormalized, x_loss.unsqueeze(1), y_loss.unsqueeze(1), force_loss.unsqueeze(1), touch_loss.unsqueeze(1)), axis=1)
        dataframe_temp = pd.DataFrame(data_row.data.cpu())
        dataframe_temp.columns = column_names  # Set the column names
        dataframe = pd.concat([dataframe, dataframe_temp])
# Save the Pandas DataFrame to an Excel file
dataframe.to_excel("../../../Data/Results/Single_Point_Contact_Estimation_Results/test_error_results.xlsx", index=False)